# Pandas: Filter · GroupBy · Aggregation · SQL
## Complete Tutorial with Examples

Each section stands alone — jump to what you need.
Run cells with **Shift + Enter**.

| Section | Topic |
|---|---|
| 1 | Setup & Dataset |
| 2 | **Filter** — 10 techniques |
| 3 | **GroupBy** — 8 techniques |
| 4 | **Aggregation** — built-in + custom |
| 5 | **SQL on DataFrames** — WHERE, GROUP BY, JOIN, HAVING, subqueries, window functions |
| 6 | Side-by-side comparison: pandas vs SQL |

---

## 1 · Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
from pandasql import sqldf   # pip install pandasql

# Helper so we don't repeat locals() every time
sql = lambda q: sqldf(q, {'df': df, 'dept': dept})

print("Libraries loaded ✓")

In [ ]:
# ── Main dataset: EMR cluster chargeback ────────────────────────
df = pd.DataFrame({
    'employee_id': ['C1234','F5678','C9012','F3456','C7890','C1111','F2222'],
    'department':  ['Quant','Risk','Quant','Tech','Risk','Quant','Tech'],
    'month':       ['2025-01','2025-01','2025-02','2025-02','2025-02','2025-03','2025-03'],
    'cost':        [1200.0, 850.0, 2100.0, 450.0, 3200.0, 1800.0, 600.0],
    'cpu_hours':   [120, 80, 210, 45, 310, 180, 60],
    'gpu_hours':   [10,  0,  30,  5,  50,  20,  0],
    'job_count':   [15,  8,  22,  5,  30,  18,  7],
})

# ── Lookup table: department info ────────────────────────────────
dept = pd.DataFrame({
    'department': ['Quant', 'Risk',  'Tech'],
    'vp':         ['Alice', 'Bob',   'Carol'],
    'budget':     [500000,  300000,  200000],
})

print(f"df shape: {df.shape}")
df

In [ ]:
dept

## 2 · Filter

Filtering selects **rows** that match a condition.
All filters return a new DataFrame — the original is unchanged.

---

### 2.1 — Single condition

In [ ]:
# Rows where cost > 1000
df[df['cost'] > 1000]

### 2.2 — AND condition  ( & )

In [ ]:
# Quant department AND cost > 1000
# ⚠️ Each condition MUST be wrapped in parentheses
df[(df['department'] == 'Quant') & (df['cost'] > 1000)]

### 2.3 — OR condition  ( | )

In [ ]:
# Very cheap (< 500) OR very expensive (> 2000)
df[(df['cost'] < 500) | (df['cost'] > 2000)]

### 2.4 — NOT condition  ( ~ )

In [ ]:
# Exclude Tech department
df[~(df['department'] == 'Tech')]

# Same using != 
# df[df['department'] != 'Tech']

### 2.5 — Filter on a list of values  ( isin )

In [ ]:
# Keep only Quant and Risk
df[df['department'].isin(['Quant', 'Risk'])]

### 2.6 — Range filter  ( between )

In [ ]:
# Cost between 800 and 2000 (inclusive)
df[df['cost'].between(800, 2000)]

### 2.7 — String filters

In [ ]:
# Employee IDs starting with C
df[df['employee_id'].str.startswith('C')]

In [ ]:
# Employee IDs containing '123'
df[df['employee_id'].str.contains('123')]

### 2.8 — query()  — SQL-like string syntax

In [ ]:
# Cleaner for complex conditions — reads like SQL WHERE clause
df.query("department == 'Quant' and cost > 1000")

In [ ]:
# Use @ to reference a Python variable inside query
min_cost = 1000
df.query("cost > @min_cost and department != 'Tech'")

### 2.9 — Top N and Bottom N rows

In [ ]:
# 3 most expensive users
df.nlargest(3, 'cost')

In [ ]:
# 2 cheapest users
df.nsmallest(2, 'cost')

### 2.10 — Filter on multiple columns combined

In [ ]:
# C-employees in Quant, Feb only
df[
    df['employee_id'].str.startswith('C') &
    (df['department'] == 'Quant') &
    (df['month'] == '2025-02')
]

### Filter Quick Reference

| Goal | Code |
|---|---|
| equals | `df[df['col'] == value]` |
| not equal | `df[df['col'] != value]` |
| greater / less | `df[df['col'] > value]` |
| AND | `df[(cond1) & (cond2)]` |
| OR | `df[(cond1) \| (cond2)]` |
| NOT | `df[~cond]` |
| in a list | `df[df['col'].isin([a, b])]` |
| range | `df[df['col'].between(lo, hi)]` |
| string starts with | `df[df['col'].str.startswith('X')]` |
| SQL-style | `df.query("col > 100 and other == 'A'")` |
| top N | `df.nlargest(N, 'col')` |

---

## 3 · GroupBy

GroupBy splits the DataFrame into groups, applies a function to each,
then combines the results.

```
Split → Apply → Combine
```
---

### 3.1 — Basic groupby: one column

In [ ]:
# Total cost per department
df.groupby('department')['cost'].sum()

### 3.2 — Groupby multiple columns

In [ ]:
# Total cost per department per month
df.groupby(['department', 'month'])['cost'].sum().reset_index()

### 3.3 — Multiple aggregations with agg()

In [ ]:
df.groupby('department').agg(
    total_cost  = ('cost',        'sum'),
    avg_cost    = ('cost',        'mean'),
    max_cost    = ('cost',        'max'),
    min_cost    = ('cost',        'min'),
    std_cost    = ('cost',        'std'),
    num_users   = ('employee_id', 'nunique'),
    total_jobs  = ('job_count',   'sum'),
).round(2)

### 3.4 — transform()  — adds result back as a new column

In [ ]:
# Useful for: pct of group, z-score within group, fill with group mean
df2 = df.copy()

# Add each user's dept total alongside their row
df2['dept_total']    = df2.groupby('department')['cost'].transform('sum')

# Each user's % share within their department
df2['pct_of_dept']   = (df2['cost'] / df2['dept_total'] * 100).round(1)

# Z-score within department (how many std devs from dept mean)
df2['cost_zscore']   = df2.groupby('department')['cost'].transform(
    lambda x: ((x - x.mean()) / x.std()).round(2)
)

df2[['employee_id','department','cost','dept_total','pct_of_dept','cost_zscore']]

### 3.5 — filter()  — keep only groups that pass a condition

In [ ]:
# Keep only departments where total cost > 2000
df.groupby('department').filter(lambda g: g['cost'].sum() > 2000)

### 3.6 — Cumulative sum within group

In [ ]:
# Running total cost per department, sorted by month
df_sorted = df.sort_values(['department', 'month']).copy()
df_sorted['running_cost'] = df_sorted.groupby('department')['cost'].cumsum()
df_sorted[['employee_id','department','month','cost','running_cost']]

### 3.7 — Rank within group

In [ ]:
# Rank each user by cost within their department (1 = highest)
df2 = df.copy()
df2['rank_in_dept'] = df2.groupby('department')['cost']                          .rank(ascending=False)                          .astype(int)
df2[['employee_id','department','cost','rank_in_dept']]    .sort_values(['department','rank_in_dept'])

### 3.8 — Apply a custom function to each group

In [ ]:
# For each department: show min, max, and range
def dept_summary(group):
    return pd.Series({
        'min_cost'  : group['cost'].min(),
        'max_cost'  : group['cost'].max(),
        'range_cost': group['cost'].max() - group['cost'].min(),
        'top_user'  : group.loc[group['cost'].idxmax(), 'employee_id'],
    })

df.groupby('department').apply(dept_summary, include_groups=False)

### GroupBy Quick Reference

| Goal | Code |
|---|---|
| Sum by group | `df.groupby('col')['val'].sum()` |
| Multiple metrics | `df.groupby('col').agg(n=('val','sum'), ...)` |
| Multiple group cols | `df.groupby(['a','b'])['val'].sum()` |
| Add back to df | `df.groupby('col')['val'].transform('sum')` |
| % of group | `x / groupby.transform('sum') * 100` |
| Keep passing groups | `.groupby().filter(lambda g: ...)` |
| Running total | `.groupby().transform(lambda x: x.cumsum())` |
| Rank in group | `.groupby()['col'].rank(ascending=False)` |

---

## 4 · Aggregation

Aggregation **reduces** many values to one summary value.

---

### 4.1 — Built-in single aggregations

In [ ]:
print(f"Sum    : {df['cost'].sum():,.0f}")
print(f"Mean   : {df['cost'].mean():,.1f}")
print(f"Median : {df['cost'].median():,.1f}")
print(f"Std    : {df['cost'].std():,.1f}")
print(f"Min    : {df['cost'].min():,.0f}")
print(f"Max    : {df['cost'].max():,.0f}")
print(f"Count  : {df['cost'].count()}")

### 4.2 — Many aggregations at once

In [ ]:
df['cost'].agg(['sum','mean','median','std','min','max','count']).round(2)

### 4.3 — Custom aggregation function

In [ ]:
# Define your own aggregation
def pct_above_1000(series):
    """What % of values are above 1000?"""
    return round((series > 1000).mean() * 100, 1)

def coefficient_of_variation(series):
    """Std / Mean — measures relative variability"""
    return round(series.std() / series.mean() * 100, 1)

df['cost'].agg([
    'sum',
    'mean',
    pct_above_1000,
    coefficient_of_variation,
])

### 4.4 — Different aggregations per column

In [ ]:
df.groupby('department').agg({
    'cost'      : ['sum', 'mean', 'max'],
    'cpu_hours' : ['sum', 'mean'],
    'job_count' : 'sum',
}).round(2)

### 4.5 — Percentiles / quantiles

In [ ]:
df.groupby('department')['cost'].agg(
    p25 = lambda x: x.quantile(0.25),
    p50 = lambda x: x.quantile(0.50),
    p75 = lambda x: x.quantile(0.75),
    iqr = lambda x: x.quantile(0.75) - x.quantile(0.25),
).round(2)

### 4.6 — Pivot table with totals

In [ ]:
# Pivot table = groupby displayed as a grid, with row/col totals
pd.pivot_table(
    df,
    index      = 'department',   # rows
    columns    = 'month',        # columns
    values     = 'cost',         # values
    aggfunc    = 'sum',          # aggregation
    fill_value = 0,
    margins    = True,           # add Total row and column
    margins_name = 'TOTAL',
)

### 4.7 — Cross-tabulation (frequency table)

In [ ]:
# Count of users per department per month
pd.crosstab(df['department'], df['month'], margins=True)

In [ ]:
# Sum of cost per department per month (crosstab with values)
pd.crosstab(
    df['department'],
    df['month'],
    values=df['cost'],
    aggfunc='sum',
    margins=True,
).round(0)

### Aggregation Function Reference

| Function | What it returns |
|---|---|
| `sum` | total |
| `mean` | average |
| `median` | middle value (robust to outliers) |
| `std` | standard deviation |
| `var` | variance |
| `min` / `max` | smallest / largest |
| `count` | non-null count |
| `nunique` | number of distinct values |
| `first` / `last` | first or last value |
| `quantile(0.25)` | 25th percentile |
| custom function | anything you write |

---

## 5 · SQL on DataFrames

`pandasql` lets you write real SQL against pandas DataFrames.
Great if you already know SQL and want to use it directly.

```bash
pip install pandasql
```
---

### 5.1 — Setup

In [ ]:
from pandasql import sqldf

# sql() runs a query against df and dept (defined in section 1)
sql = lambda q: sqldf(q, {'df': df, 'dept': dept})

print("pandasql ready ✓")

### 5.2 — SELECT and WHERE (= pandas filter)

In [ ]:
sql("""
    SELECT employee_id, department, month, cost
    FROM   df
    WHERE  cost > 1000
    ORDER  BY cost DESC
""")

### 5.3 — GROUP BY and aggregate functions

In [ ]:
sql("""
    SELECT
        department,
        SUM(cost)      AS total_cost,
        AVG(cost)      AS avg_cost,
        MAX(cost)      AS max_cost,
        COUNT(*)       AS num_rows,
        COUNT(DISTINCT employee_id) AS num_users
    FROM   df
    GROUP  BY department
    ORDER  BY total_cost DESC
""")

### 5.4 — HAVING (= filter AFTER grouping)

In [ ]:
# HAVING filters on aggregated values — WHERE cannot do this
sql("""
    SELECT   department, SUM(cost) AS total_cost
    FROM     df
    GROUP BY department
    HAVING   SUM(cost) > 3000
""")

### 5.5 — JOIN (= pandas merge)

In [ ]:
# LEFT JOIN — keep all df rows, add dept info where it matches
sql("""
    SELECT
        df.employee_id,
        df.department,
        dept.vp,
        dept.budget,
        df.cost,
        ROUND(df.cost * 100.0 / dept.budget, 2) AS pct_of_budget
    FROM   df
    LEFT   JOIN dept ON df.department = dept.department
    ORDER  BY df.cost DESC
""")

### 5.6 — Subquery (= filter using a computed value)

In [ ]:
# Users whose cost is above the overall average
sql("""
    SELECT employee_id, department, cost
    FROM   df
    WHERE  cost > (SELECT AVG(cost) FROM df)
    ORDER  BY cost DESC
""")

In [ ]:
# Users who are the top spender in their department
sql("""
    SELECT d.*
    FROM   df d
    WHERE  cost = (
        SELECT MAX(cost)
        FROM   df
        WHERE  department = d.department
    )
""")

### 5.7 — CASE WHEN (= np.where / apply)

In [ ]:
sql("""
    SELECT
        employee_id,
        department,
        cost,
        CASE
            WHEN cost >= 2000 THEN 'High'
            WHEN cost >= 1000 THEN 'Mid'
            ELSE                   'Low'
        END AS cost_band
    FROM df
    ORDER BY cost DESC
""")

### 5.8 — Window functions — RANK, OVER, PARTITION BY

In [ ]:
# Rank each user by cost within their department
sql("""
    SELECT
        employee_id,
        department,
        cost,
        RANK() OVER (
            PARTITION BY department
            ORDER BY cost DESC
        ) AS dept_rank
    FROM df
    ORDER BY department, dept_rank
""")

In [ ]:
# Running total by department (ORDER matters here)
sql("""
    SELECT
        employee_id,
        department,
        month,
        cost,
        SUM(cost) OVER (
            PARTITION BY department
            ORDER BY month
        ) AS running_cost
    FROM df
    ORDER BY department, month
""")

### 5.9 — GROUP BY with ROLLUP (subtotals)

In [ ]:
# Monthly totals + grand total in one query
sql("""
    SELECT
        COALESCE(department, 'ALL DEPTS') AS department,
        SUM(cost)  AS total_cost,
        COUNT(*)   AS rows
    FROM   df
    GROUP  BY department

    UNION ALL

    SELECT 'GRAND TOTAL', SUM(cost), COUNT(*)
    FROM   df
""")

## 6 · Pandas vs SQL — Side-by-Side

Every common SQL operation has a direct pandas equivalent.

---

In [ ]:
# ── SELECT columns ──────────────────────────────────────────────
# SQL:    SELECT employee_id, cost FROM df
# Pandas:
df[['employee_id', 'cost']].head(3)

In [ ]:
# ── WHERE ───────────────────────────────────────────────────────
# SQL:    SELECT * FROM df WHERE cost > 1000
# Pandas:
df[df['cost'] > 1000][['employee_id','department','cost']]

In [ ]:
# ── GROUP BY + aggregate ────────────────────────────────────────
# SQL:    SELECT department, SUM(cost) FROM df GROUP BY department
# Pandas:
df.groupby('department')['cost'].sum().reset_index()

In [ ]:
# ── HAVING ──────────────────────────────────────────────────────
# SQL:    ... GROUP BY department HAVING SUM(cost) > 3000
# Pandas:
result = df.groupby('department')['cost'].sum().reset_index()
result[result['cost'] > 3000]

In [ ]:
# ── ORDER BY ────────────────────────────────────────────────────
# SQL:    SELECT * FROM df ORDER BY cost DESC LIMIT 3
# Pandas:
df.sort_values('cost', ascending=False).head(3)[['employee_id','cost']]

In [ ]:
# ── JOIN ────────────────────────────────────────────────────────
# SQL:    SELECT * FROM df LEFT JOIN dept ON df.department = dept.department
# Pandas:
df.merge(dept, on='department', how='left')[['employee_id','department','vp','cost']]

In [ ]:
# ── DISTINCT ────────────────────────────────────────────────────
# SQL:    SELECT DISTINCT department FROM df
# Pandas:
pd.DataFrame({'department': df['department'].unique()})

In [ ]:
# ── COUNT DISTINCT ──────────────────────────────────────────────
# SQL:    SELECT COUNT(DISTINCT employee_id) FROM df
# Pandas:
df['employee_id'].nunique()

In [ ]:
# ── CASE WHEN ───────────────────────────────────────────────────
# SQL:    CASE WHEN cost >= 2000 THEN 'High' WHEN cost >= 1000 THEN 'Mid' ELSE 'Low'
# Pandas (np.where for 2 outcomes, apply/cut for more):
import numpy as np
df['band'] = pd.cut(
    df['cost'],
    bins=[0, 999, 1999, float('inf')],
    labels=['Low','Mid','High']
)
df[['employee_id','cost','band']]

In [ ]:
# ── SUBQUERY ────────────────────────────────────────────────────
# SQL:    WHERE cost > (SELECT AVG(cost) FROM df)
# Pandas:
avg = df['cost'].mean()
df[df['cost'] > avg][['employee_id','cost']]

In [ ]:
# ── WINDOW RANK ─────────────────────────────────────────────────
# SQL:    RANK() OVER (PARTITION BY department ORDER BY cost DESC)
# Pandas:
df['dept_rank'] = df.groupby('department')['cost']                     .rank(ascending=False, method='min').astype(int)
df[['employee_id','department','cost','dept_rank']].sort_values(['department','dept_rank'])

---
## Summary Cheat Sheet

### Filter
```python
df[df['col'] > value]                          # single condition
df[(df['a'] > 1) & (df['b'] == 'X')]           # AND
df[(df['a'] < 1) | (df['b'] == 'X')]           # OR
df[~(df['col'] == 'X')]                        # NOT
df[df['col'].isin(['A','B'])]                  # IN list
df[df['col'].between(lo, hi)]                  # range
df.query("col > 100 and other == 'A'")         # SQL style
df.nlargest(N, 'col')                          # top N
```

### GroupBy
```python
df.groupby('col')['val'].sum()                 # simple
df.groupby('col').agg(n=('val','sum'), ...)    # named agg
df.groupby('col')['val'].transform('sum')      # add back to rows
df.groupby('col').filter(lambda g: g['val'].sum() > X)  # filter groups
df.groupby('col')['val'].rank(ascending=False) # rank in group
```

### Aggregation
```python
df['col'].agg(['sum','mean','median','std'])   # multiple at once
df.pivot_table(index, columns, values, aggfunc, margins=True)
pd.crosstab(df['a'], df['b'], values=df['c'], aggfunc='sum')
```

### SQL (pandasql)
```python
from pandasql import sqldf
sql = lambda q: sqldf(q, {'df': df, 'dept': dept})
sql("SELECT ... FROM df WHERE ... GROUP BY ... HAVING ... ORDER BY ...")
```

### Pandas ↔ SQL mapping
| SQL | Pandas |
|---|---|
| `WHERE` | `df[condition]` or `.query()` |
| `GROUP BY + agg` | `.groupby().agg()` |
| `HAVING` | filter after groupby |
| `ORDER BY` | `.sort_values()` |
| `LIMIT` | `.head(N)` |
| `JOIN` | `.merge()` |
| `DISTINCT` | `.unique()` / `.nunique()` |
| `CASE WHEN` | `pd.cut()` / `np.where()` / `.apply()` |
| `RANK() OVER (PARTITION BY)` | `.groupby().rank()` |
| Subquery | compute value first, then filter |